# CS604 Project - 02 Train and submit

Thin runner: all logic is in `common/` and `methods/` in the repo. Pick a config, run top to bottom.
Use an **A100** Colab GPU for real runs (a T4 is ~25 img/s: unusable). Try `smoke` first on any GPU - the code has not been run on a GPU yet.

In [ ]:
REPO_URL = "https://github.com/notzdean/CS604-Computer-Vision.git"   # <- set once
CONFIG = "smoke"        # a file in configs/ without .yaml: smoke | full_ft_baseline | lora_baseline | bitfit_baseline | your own
WHO    = "yourname"     # goes into results.csv
TEAM   = "XX"           # team number for the submission file name (val_team_XX.txt)
EXTRA  = ""             # optional overrides, e.g. "lr=1e-4 label_smoothing=0.1"


## 1. Code

In [ ]:
%%bash -s "$REPO_URL"
if [ -d /content/repo/.git ]; then git -C /content/repo pull; else git clone "$1" /content/repo; fi
pip install -q -r /content/repo/requirements.txt

## 2. Data (skipped if already there)

In [ ]:
%%bash
set -e
set -o pipefail
D=/content/cs604-course-data
U=https://huggingface.co/datasets/doem1997/cs604-course-data/resolve/main
if [ -f $D/.ready ]; then echo "Data already prepared - skipping download."; exit 0; fi
mkdir -p $D && cd $D
curl -sSfL $U/train.txt -o train.txt
curl -sSfL $U/val.txt -o val.txt
# stream both archives in parallel, extracting as they download
curl -sSfL $U/val.tar.gz   | tar -xz &
P1=$!
curl -sSfL $U/train.tar.gz | tar -xz
wait $P1
touch .ready
df -h /content
ls $D

## 3. Where to save runs
Google Drive if it can be mounted (survives disconnects; needs only a few hundred MB per run), otherwise the VM disk.

In [ ]:
import os
try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/cs604_runs"
except Exception as e:
    print("Drive not mounted, using the VM disk (lost on disconnect):", e)
    OUT = "/content/runs"
os.makedirs(OUT, exist_ok=True)
%cd /content/repo
print("runs go to", OUT)

## 4. Train
Resumable: if the session dies, run this cell again with the same CONFIG and it continues from the last checkpoint.

In [ ]:
!python -m common.train configs/{CONFIG}.yaml out_dir=$OUT/$CONFIG who=$WHO results_csv=$OUT/results.csv $EXTRA

## 5. Result row
Copy the last row of this file into `results/results.csv` in your local clone and commit it (the Colab VM cannot push).

In [ ]:
!tail -n 2 $OUT/results.csv

## 6. Submission file (only when the hold-out score looks good - max 5 per day)
Writes and validates `val_team_XX.txt` and its zip inside the run folder. Upload the zip to CodaBench.

In [ ]:
!python -m common.submit $OUT/$CONFIG --team $TEAM --split val